In [ ]:
# Basis-Setup: Imports und Anzeigeoptionen
import os
import pandas as pd
from sqlalchemy import create_engine, text
import psycopg2
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

# Repo-Tabelle laden und Basis-Metriken
Dieses Notebook ist ein **simples, transformierbares Grundgerüst**.

Ablauf:
1. DB-Verbindung aus Umgebungsvariable herstellen
2. `repo`-Tabelle laden
3. Datenqualität kurz prüfen
4. Erste Metriken berechnen

In [ ]:
# 1) Build connection (Docker-Compose compatible)
# Priority: full URL from ENV -> otherwise build from individual values

DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")        # docker-compose maps 5432:5432
DB_NAME = os.getenv("DB_NAME", "github_events")
DB_USER = os.getenv("DB_USER", "github")
DB_PASSWORD = os.getenv("DB_PASSWORD", "github_secret")

DATABASE_URL = os.getenv("DATABASE_URL") or (
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

engine = create_engine(DATABASE_URL)
print(f"DB connection ready: {DB_HOST}:{DB_PORT}/{DB_NAME}")

In [ ]:
# 2) Daten aus Repo-Tabelle laden (auto-detect: repos oder repo)
with engine.connect() as conn:
    table_names = pd.read_sql(
        text("SELECT tablename FROM pg_catalog.pg_tables WHERE schemaname = 'public' ORDER BY tablename"),
        conn,
    )["tablename"].tolist()

if "repos" in table_names:
    source_table = "repos"
elif "repo" in table_names:
    source_table = "repo"
else:
    raise ValueError(
        "Keine Repo-Tabelle gefunden. Erwartet 'repos' oder 'repo'. "
        f"Vorhandene Tabellen: {table_names}"
    )

query = f"SELECT * FROM {source_table}"
with engine.connect() as conn:
    repo_df = pd.read_sql(text(query), conn)

print(f"Quelle: {source_table} | Geladene Zeilen: {len(repo_df):,}")
repo_df.head()

In [ ]:
# 3) Quick Data Checks
repo_df.info()

nulls = repo_df.isna().sum().sort_values(ascending=False)
nulls[nulls > 0].head(20)

In [ ]:
# 4) Basis-Measures (anpassbar an deine Spalten)
id_col = "id" if "id" in repo_df.columns else ("repo_id" if "repo_id" in repo_df.columns else None)
name_col = "name" if "name" in repo_df.columns else ("full_name" if "full_name" in repo_df.columns else None)

measures = {
    "rows": len(repo_df),
    "columns": repo_df.shape[1],
    "distinct_repo_ids": repo_df[id_col].nunique() if id_col else None,
    "distinct_repo_names": repo_df[name_col].nunique() if name_col else None,
}

pd.Series(measures, name="value")

In [ ]:
# 5) Transformations-Template (intuitiv & wiederverwendbar)
def transform_repo(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # Beispiel: Standardisierte Spaltennamen
    out.columns = [c.strip().lower() for c in out.columns]

    # Beispiel: Datumsfelder robust casten (nur falls vorhanden)
    for col in ["created_at", "updated_at", "pushed_at"]:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce", utc=True)

    return out

repo_t = transform_repo(repo_df)
repo_t.head()